In [ ]:
from importlib.metadata import version
from pathlib import Path
from docling.datamodel.accelerator_options import AcceleratorDevice, AcceleratorOptions
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import (
    HeadingHierarchyOptions, 
    OcrAutoOptions, 
    PdfPipelineOptions,
    TableFormerMode, 
    TableStructureOptions,
)
from docling.document_converter import (
    DocumentConverter, 
    PdfFormatOption,
)

print("docling:", version("docling"))
print("docling-core:", version("docling-core"))

In [ ]:
# AcceleratorOptions
accelerator_options = AcceleratorOptions(
    device=AcceleratorDevice.CUDA,
    num_threads=4,
    cuda_use_flash_attention2=False,
)


In [ ]:
# OcrAutoOptions
ocr_options = OcrAutoOptions(lang=["en", "ja"], scale=3.0)


In [ ]:
# TableStructureOptions
table_options = TableStructureOptions(
    mode=TableFormerMode.ACCURATE,
    do_cell_matching=True,
)


In [ ]:
# HeadingHierarchyOptions
heading_options = HeadingHierarchyOptions(
    enabled=True, use_bookmarks=True, use_numbering=True,
    use_style=True, use_font_style=True, max_level=6,
)


In [ ]:
# PdfPipelineOptions
pdf_options = PdfPipelineOptions(
    # Runtime & security
    document_timeout=None,
    accelerator_options=accelerator_options,
    enable_remote_services=False,
    allow_external_plugins=True,
    artifacts_path=None,  # Đổi thành Path("...") khi dùng model tải sẵn

    # OCR, layout & table structure
    do_ocr=True,
    ocr_options=ocr_options,
    do_table_structure=True,
    table_structure_options=table_options,
    force_backend_text=False,

    # Reading order / heading hierarchy
    use_reading_order_separators=True,
    heading_hierarchy_options=heading_options,

    # Assets cho validator, crop & recovery
    images_scale=2.0,
    generate_page_images=True,
    generate_picture_images=True,
    generate_table_images=True,
    generate_parsed_pages=True,

    # Built-in enrichment
    do_picture_classification=True,
    do_picture_description=False,
    do_chart_extraction=True,
    do_code_enrichment=True,
    do_formula_enrichment=True,

    # Batching / throughput
    ocr_batch_size=2,
    layout_batch_size=2,
    table_batch_size=2,
    batch_polling_interval_seconds=0.5,
    queue_max_size=100,
    stage_shutdown_timeout_seconds=60.0,
)

# pdf_options.layout_options
# pdf_options.picture_classification_options
# pdf_options.picture_description_options
# pdf_options.chart_extraction_options
# pdf_options.code_formula_options

In [ ]:
converter = DocumentConverter(
    allowed_formats=[
        InputFormat.PDF, InputFormat.DOCX,
        InputFormat.PPTX, InputFormat.XLSX,
    ],
    format_options={
        InputFormat.PDF: PdfFormatOption(pipeline_options=pdf_options),
    },
)